In [ ]:
import os
import sys
import pandas as pd
import os

os.chdir("..")

print('current working dir:',os.getcwd())

print("Imports completed successfully.")

In [ ]:
import os
os.chdir(r"d:\PROJECTS\RecommendIQ")
print("Current Working Directory:")
print(os.getcwd())

In [ ]:


from src import data_preprocessing
from src import feature_engineering # type: ignore
from src import segmentation
from src import recommender
from src import feedback
from src.personalization import PersonalizationEngine

print("All src modules imported successfully.")

In [ ]:
from sqlalchemy import text
from src.database import get_engine

print("=" * 60)
print("Testing database.py")
print("=" * 60)

try:
    engine = get_engine()

    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))

        print(result.scalar())

    print("✅ Database connection successful.")

except Exception as e:
    print("❌ Database connection failed.")
    print(e)

In [ ]:
print("=" * 70)
print("Testing data_preprocessing.py")
print("=" * 70)

from src import data_preprocessing

try:

    # --------------------------------------------------
    # Load cleaned tables from MySQL
    # --------------------------------------------------

    events_df = pd.read_sql(
        "SELECT * FROM events_cleaned limit 5",
        engine
    )

    category_tree_df = pd.read_sql(
        "SELECT * FROM category_tree_cleaned limit 5 ",
        engine
    )

    item_properties_df = pd.read_sql(
        "SELECT * FROM item_properties_cleaned LIMIT 5",
        engine
    )

    print("\n✅ Tables loaded successfully.")

    # --------------------------------------------------
    # Test convert_timestamp()
    # --------------------------------------------------

    events_df = data_preprocessing.convert_timestamp(events_df)

    print("✅ convert_timestamp() passed")

    # --------------------------------------------------
    # Test remove_duplicates()
    # --------------------------------------------------

    events_df = data_preprocessing.remove_duplicates(events_df)

    print("✅ remove_duplicates() passed")

    # --------------------------------------------------
    # Test check_missing_values()
    # --------------------------------------------------

    missing_summary = data_preprocessing.check_missing_values(
        events_df
    )

    print("✅ check_missing_values() passed")

    display(missing_summary)

    # --------------------------------------------------
    # Test validate_transaction_ids()
    # --------------------------------------------------

    data_preprocessing.validate_transaction_ids(events_df)

    print("✅ validate_transaction_ids() passed")

    # --------------------------------------------------
    # Test clean_category_tree()
    # --------------------------------------------------

    category_tree_df = data_preprocessing.clean_category_tree(
        category_tree_df
    )

    print("✅ clean_category_tree() passed")

    # --------------------------------------------------
    # Test merge_item_properties()
    # --------------------------------------------------

    merged_items = data_preprocessing.merge_item_properties(
        item_properties_df.iloc[:50000],
        item_properties_df.iloc[50000:]
    )

    print("✅ merge_item_properties() passed")

    # --------------------------------------------------
    # Test validate_dataset()
    # --------------------------------------------------

    data_preprocessing.validate_dataset(
        events_df,
        "Events"
    )

    data_preprocessing.validate_dataset(
        merged_items,
        "Item Properties"
    )

    data_preprocessing.validate_dataset(
        category_tree_df,
        "Category Tree"
    )

    print("\n")

    print("=" * 70)
    print("✅ data_preprocessing.py PASSED")
    print("=" * 70)

except Exception as e:

    print("=" * 70)
    print("❌ data_preprocessing.py FAILED")
    print("=" * 70)

    print(e)

In [ ]:
print("=" * 70)
print("Testing feature_engineering.py")
print("=" * 70)

from src import feature_engineering

try:

    # --------------------------------------------------
    # Create Interaction Strength
    # --------------------------------------------------

    events_features = feature_engineering.create_interaction_strength(
        events_df.copy()
    )

    print("✅ create_interaction_strength() passed")

    # --------------------------------------------------
    # Create Recency Feature
    # --------------------------------------------------

    events_features = feature_engineering.create_recency_feature(
        events_features
    )

    print("✅ create_recency_feature() passed")

    # --------------------------------------------------
    # Create User Features
    # --------------------------------------------------

    user_profiles = feature_engineering.create_user_features(
        events_features
    )

    print("✅ create_user_features() passed")

    # --------------------------------------------------
    # Create Item Features
    # --------------------------------------------------

    item_profiles = feature_engineering.create_item_features(
        events_features
    )

    print("✅ create_item_features() passed")

    # --------------------------------------------------
    # Create User-Item Features
    # --------------------------------------------------

    user_item_features = (
        feature_engineering.create_user_item_features(
            events_features
        )
    )

    print("✅ create_user_item_features() passed")

    # --------------------------------------------------
    # Test Complete Pipeline
    # --------------------------------------------------

    customer_features = (
        feature_engineering.build_feature_pipeline(
            events_df.copy()
        )
    )

    print("✅ build_feature_pipeline() passed")

    # --------------------------------------------------
    # Display Outputs
    # --------------------------------------------------

    print("\nCustomer Features")

    display(customer_features.head())

    print(customer_features.shape)

    print("\nUser Profiles")

    display(user_profiles.head())

    print(user_profiles.shape)

    print("\nItem Profiles")

    display(item_profiles.head())

    print(item_profiles.shape)

    print("\nUser Item Features")

    display(user_item_features.head())

    print(user_item_features.shape)

    # --------------------------------------------------
    # Save Feature Files 
    # --------------------------------------------------

    customer_features.to_csv(
        "data/features/customer_features.csv",
        index=False
    )

    user_profiles.to_csv(
        "data/features/user_profiles.csv",
        index=False
    )

    user_item_features.to_csv(
        "data/features/user_item_features.csv",
        index=False
    )

    print("\nFeature files saved successfully.")

    print("\n" + "=" * 70)
    print("✅ feature_engineering.py PASSED")
    print("=" * 70)

except Exception as e:

    print("\n" + "=" * 70)
    print("❌ feature_engineering.py FAILED")
    print("=" * 70)

    print(e)

In [ ]:
print("=" * 70)
print("Testing segmentation.py")
print("=" * 70)

from src import segmentation

try:

    # --------------------------------------------------
    # Load Customer Features
    # --------------------------------------------------

    customer_df = segmentation.load_customer_features(
        "data/features/customer_features.csv"
    )

    print("✅ load_customer_features() passed")

    # --------------------------------------------------
    # Preprocess Features
    # --------------------------------------------------

    X_scaled, scaler = segmentation.preprocess_features(
        customer_df
    )

    print("✅ preprocess_features() passed")

    # --------------------------------------------------
    # Train KMeans Model
    # --------------------------------------------------

    kmeans = segmentation.train_kmeans(
        X_scaled,
        n_clusters=5
    )

    print("✅ train_kmeans() passed")

    # --------------------------------------------------
    # Assign Clusters
    # --------------------------------------------------

    segmented_df = segmentation.assign_clusters(
        customer_df,
        kmeans,
        X_scaled
    )

    print("✅ assign_clusters() passed")

    # --------------------------------------------------
    # Map Cluster Names
    # --------------------------------------------------

    segmented_df = segmentation.map_cluster_names(
        segmented_df
    )

    print("✅ map_cluster_names() passed")

    # --------------------------------------------------
    # Predict Segment for First Customer
    # --------------------------------------------------

    sample_customer = customer_df.drop(
        columns=["visitorid"]
    ).head(1)

    cluster, segment = segmentation.predict_customer_segment(
        sample_customer,
        scaler,
        kmeans
    )

    print(f"Predicted Cluster : {cluster}")
    print(f"Predicted Segment : {segment}")

    print("✅ predict_customer_segment() passed")

    # --------------------------------------------------
    # Generate Summary
    # --------------------------------------------------

    summary = segmentation.segment_summary(
        segmented_df
    )

    print("✅ segment_summary() passed")

    # --------------------------------------------------
    # Display Results
    # --------------------------------------------------

    print("\nSegmented Dataset")
    display(segmented_df.head())
    print(segmented_df.shape)

    print("\nSegment Summary")
    display(summary)

    print("\n" + "=" * 70)
    print("✅ segmentation.py PASSED")
    print("=" * 70)

except Exception as e:

    print("\n" + "=" * 70)
    print("❌ segmentation.py FAILED")
    print("=" * 70)
    print(e)

In [ ]:
print("=" * 70)
print("Testing segmentation.py")
print("=" * 70)

from src import segmentation

try:

    # --------------------------------------------------
    # Load Customer Features
    # --------------------------------------------------

    customer_df = segmentation.load_customer_features(
        "data/features/customer_features.csv"
    )

    print("✅ load_customer_features() passed")

    # --------------------------------------------------
    # Preprocess Features
    # --------------------------------------------------

    X_scaled, scaler = segmentation.preprocess_features(
        customer_df
    )

    print("✅ preprocess_features() passed")

    # --------------------------------------------------
    # Train KMeans Model
    # --------------------------------------------------

    kmeans = segmentation.train_kmeans(
        X_scaled,
        n_clusters=5
    )

    print("✅ train_kmeans() passed")

    # --------------------------------------------------
    # Assign Clusters
    # --------------------------------------------------

    segmented_df = segmentation.assign_clusters(
        customer_df,
        kmeans,
        X_scaled
    )

    print("✅ assign_clusters() passed")

    # --------------------------------------------------
    # Map Cluster Names
    # --------------------------------------------------

    segmented_df = segmentation.map_cluster_names(
        segmented_df
    )

    print("✅ map_cluster_names() passed")

    # --------------------------------------------------
    # Predict Segment for First Customer
    # --------------------------------------------------

    sample_customer = customer_df.drop(
        columns=["visitorid"]
    ).head(1)

    cluster, segment = segmentation.predict_customer_segment(
        sample_customer,
        scaler,
        kmeans
    )

    print(f"Predicted Cluster : {cluster}")
    print(f"Predicted Segment : {segment}")

    print("✅ predict_customer_segment() passed")

    # --------------------------------------------------
    # Generate Summary
    # --------------------------------------------------

    summary = segmentation.segment_summary(
        segmented_df
    )

    print("✅ segment_summary() passed")

    # --------------------------------------------------
    # Display Results
    # --------------------------------------------------

    print("\nSegmented Dataset")
    display(segmented_df.head())
    print(segmented_df.shape)

    print("\nSegment Summary")
    display(summary)

    print("\n" + "=" * 70)
    print("✅ segmentation.py PASSED")
    print("=" * 70)

except Exception as e:

    print("\n" + "=" * 70)
    print("❌ segmentation.py FAILED")
    print("=" * 70)
    print(e)

In [ ]:
print("=" * 70)
print("Testing recommender.py")
print("=" * 70)

from src import recommender

try:

    # --------------------------------------------------
    # Load Interaction Data
    # --------------------------------------------------

    interaction_df = recommender.load_interaction_data(
        "data/features/user_item_features.csv"
    )

    print("✅ load_interaction_data() passed")

    # --------------------------------------------------
    # Create Interaction Matrix
    # --------------------------------------------------

    interaction_matrix = recommender.create_interaction_matrix(
        interaction_df
    )

    print("✅ create_interaction_matrix() passed")

    # --------------------------------------------------
    # Train Similarity Model
    # --------------------------------------------------

    similarity_df = recommender.train_similarity_model(
        interaction_matrix
    )

    print("✅ train_similarity_model() passed")

    # --------------------------------------------------
    # Save Recommendation Model
    # --------------------------------------------------

    recommender.save_recommender(
        similarity_df
    )

    print("✅ save_recommender() passed")

    # --------------------------------------------------
    # Load Recommendation Model
    # --------------------------------------------------

    similarity_df = recommender.load_recommender()

    print("✅ load_recommender() passed")

    # --------------------------------------------------
    # Existing Customer Recommendation
    # --------------------------------------------------

    sample_user = interaction_df["visitorid"].iloc[0]

    existing_recommendations = (
        recommender.recommend_for_existing_user(
            visitorid=sample_user,
            interaction_df=interaction_df,
            similarity_df=similarity_df,
            top_n=10
        )
    )

    print("✅ recommend_for_existing_user() passed")

    # --------------------------------------------------
    # New Customer Recommendation
    # --------------------------------------------------

    new_user_recommendations = (
        recommender.recommend_for_new_user(
            interaction_df,
            top_n=10
        )
    )

    print("✅ recommend_for_new_user() passed")

    # --------------------------------------------------
    # Display Results
    # --------------------------------------------------

    print("\nSample Existing Customer")

    print(sample_user)

    print("\nExisting User Recommendations")

    print(existing_recommendations)

    print("\nNew User Recommendations")

    print(new_user_recommendations)

    # --------------------------------------------------
    # Save Recommendation Files
    # --------------------------------------------------

    recommendations_df = pd.DataFrame({

        "visitorid": [sample_user] * len(existing_recommendations),

        "itemid": existing_recommendations

    })

    recommendations_df.to_csv(
        "data/features/recommendations.csv",
        index=False
    )

    popular_df = pd.DataFrame({

        "itemid": new_user_recommendations

    })

    popular_df.to_csv(
        "data/features/popular_items.csv",
        index=False
    )

    print("✅ recommendations.csv saved")

    print("✅ popular_items.csv saved")

    print("\n" + "=" * 70)
    print("✅ recommender.py PASSED")
    print("=" * 70)

except Exception as e:

    print("\n" + "=" * 70)
    print("❌ recommender.py FAILED")
    print("=" * 70)

    print(e)

In [ ]:
print("=" * 70)
print("Testing feedback.py")
print("=" * 70)

from src import feedback

try:

    # --------------------------------------------------
    # Load Feedback Dataset
    # --------------------------------------------------

    feedback_df = feedback.load_feedback_data(
        "data/features/recommendations_with_feedback.csv"
    )

    print("✅ load_feedback_data() passed")

    # --------------------------------------------------
    # Preprocess Feedback
    # --------------------------------------------------

    feedback_df = feedback.preprocess_feedback(
        feedback_df
    )

    print("✅ preprocess_feedback() passed")

    # --------------------------------------------------
    # Encode Feedback
    # --------------------------------------------------

    feedback_df = feedback.encode_feedback(
        feedback_df
    )

    print("✅ encode_feedback() passed")

    # --------------------------------------------------
    # Update Recommendation Scores
    # --------------------------------------------------

    feedback_df = feedback.update_recommendation_scores(
        feedback_df
    )

    print("✅ update_recommendation_scores() passed")

    # --------------------------------------------------
    # Re-rank Recommendations
    # --------------------------------------------------

    feedback_df = feedback.rerank_recommendations(
        feedback_df
    )

    print("✅ rerank_recommendations() passed")

    # --------------------------------------------------
    # Feedback Summary
    # --------------------------------------------------

    summary = feedback.feedback_summary(
        feedback_df
    )

    print("✅ feedback_summary() passed")

    # --------------------------------------------------
    # Save Processed Feedback
    # --------------------------------------------------

    output_path = (
        "data/features/recommendations_with_feedback_processed.csv"
    )

    feedback.save_feedback_data(
        feedback_df,
        output_path
    )

    print("✅ save_feedback_data() passed")

    # --------------------------------------------------
    # Load Saved File Again
    # --------------------------------------------------

    loaded_df = feedback.load_feedback_data_file(
        output_path
    )

    print("✅ load_feedback_data_file() passed")

    # --------------------------------------------------
    # Display Results
    # --------------------------------------------------

    print("\nProcessed Feedback")

    display(loaded_df.head())

    print("\nShape")

    print(loaded_df.shape)

    print("\nColumns")

    print(loaded_df.columns.tolist())

    print("\nFeedback Distribution")

    display(summary)

    print("\n" + "=" * 70)
    print("✅ feedback.py PASSED")
    print("=" * 70)

except Exception as e:

    print("\n" + "=" * 70)
    print("❌ feedback.py FAILED")
    print("=" * 70)

    print(e)

In [ ]:
print("=" * 70)
print("Testing personalization.py")
print("=" * 70)

from src.personalization import PersonalizationEngine

try:

    # --------------------------------------------------
    # Initialize Engine
    # --------------------------------------------------

    engine = PersonalizationEngine()

    print("✅ PersonalizationEngine initialized")

    # --------------------------------------------------
    # Load Recommendations
    # --------------------------------------------------

    recommendations = engine.load_recommendations(
        "data/features/recommendations_with_feedback_processed.csv"
    )

    print("✅ load_recommendations() passed")

    # --------------------------------------------------
    # Calculate User Preferences
    # --------------------------------------------------

    user_preferences = engine.calculate_user_preferences(
        recommendations
    )

    print("✅ calculate_user_preferences() passed")

    # --------------------------------------------------
    # Calculate Personalization Scores
    # --------------------------------------------------

    personalized = engine.calculate_personalization_score(
        recommendations,
        user_preferences
    )

    print("✅ calculate_personalization_score() passed")

    # --------------------------------------------------
    # Rank Recommendations
    # --------------------------------------------------

    personalized = engine.rank_recommendations(
        personalized
    )

    print("✅ rank_recommendations() passed")

    # --------------------------------------------------
    # Save Personalized Recommendations
    # --------------------------------------------------

    output_path = (
        "data/features/personalized_recommendations_final.csv"
    )

    engine.save_personalized_recommendations(
        personalized,
        output_path
    )

    print("✅ save_personalized_recommendations() passed")

    # --------------------------------------------------
    # Test Complete Pipeline
    # --------------------------------------------------

    engine.run(
        "data/features/recommendations_with_feedback_processed.csv",
        "data/features/personalized_recommendations.csv"
    )

    print("✅ run() passed")

    # --------------------------------------------------
    # Display Results
    # --------------------------------------------------

    print("\nPersonalized Recommendations")

    display(personalized.head())

    print("\nShape")

    print(personalized.shape)

    print("\nColumns")

    print(personalized.columns.tolist())

    print("\nTop Personalized Recommendations")

    display(
        personalized[
            [
                "visitorid",
                "itemid",
                "personalization_score",
                "personalized_rank"
            ]
        ].head(10)
    )

    print("\n" + "=" * 70)
    print("✅ personalization.py PASSED")
    print("=" * 70)

except Exception as e:

    print("\n" + "=" * 70)
    print("❌ personalization.py FAILED")
    print("=" * 70)

    print(e)

# Final Validation

## Objective

This section verifies that all important project outputs were created successfully.

### Files Checked

- customer_features.csv
- user_profiles.csv
- user_item_features.csv
- customer_segments.csv
- recommendations.csv
- popular_items.csv
- recommendations_with_feedback_processed.csv
- personalized_recommendations.csv
- personalized_recommendations_final.csv
- scaler.pkl
- kmeans_model.pkl
- item_similarity.pkl

If all files exist, then the complete backend pipeline has executed successfully.

In [ ]:
import os

print("=" * 70)
print("Final Output Validation")
print("=" * 70)

files = [

    "data/features/customer_features.csv",

    "data/features/user_profiles.csv",

    "data/features/user_item_features.csv",

    "data/features/customer_segments.csv",

    "data/features/recommendations.csv",

    "data/features/popular_items.csv",

    "data/features/recommendations_with_feedback_processed.csv",

    "data/features/personalized_recommendations.csv",

    "data/features/personalized_recommendations_final.csv",

]

success = 0

for file in files:

    if os.path.exists(file):

        print(f"✅ {os.path.basename(file)}")

        success += 1

    else:

        print(f"❌ {os.path.basename(file)}")

print("\n")
print(f"Files Found : {success}/{len(files)}")

if success == len(files):

    print("\n🎉 All expected output files are present.")

else:

    print("\n⚠️ Some files are missing. Review the module that generates them.")